# Effect of pressure on flux and apparent permeability

Temperature is held at `CONDITIONS['T_operating']` and the upstream pressure is swept over `CONDITIONS['P_range']` (both from `study_config.py`).

The flux follows $J = A\,P^{\,n}$, so each fit is a straight line on log-log axes and its slope $n$ names the controlling step: $n \approx 1$ for a Henry oxide (Model 1) or the surface rate (Model 2), $n = 0.5$ for Sieverts transport. The figures show one fit per regime, with the $n = 1$ and $n = 0.5$ reference slopes labelled.

The table also gives the **net $n$** of the whole trend, $\Delta\ln J / \Delta\ln P$ between the two ends of `P_range`: the average of the local exponent $d\ln J/d\ln P$ over the range, not a fitted line, so it is reported but not drawn.

How to read the figures and tables of all four sweeps, and the difference between a regime $n$ and the net $n$: `docs/how-to/parameter-sweeps.md`.

$\Phi_{app} = J\,L/\Delta\sqrt{P}$ is computed from the same flux. Because $\Delta\sqrt{P}$ moves with the x-axis, its slope is $n - 0.5$: **flat where Sieverts transport controls** (the plateau is a true permeability) and rising as $\sqrt{P}$ where it does not. For L2a the normalisation is $\Delta P$, so its slope is $n - 1$.

## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from scipy import stats
from calculations.config.model_config import (CONDITIONS, MICROSTRUCTURE,
                                              OXIDE_DEFECTS, build_simulation_config)
from calculations.oxide_permeation import (get_oxide_properties_at_T, get_metal_properties_at_T,
                                           molecular_diffusion_flux)
from calculations.permeation_calc import calculate_simple_metal_flux, calculate_defective_metal_flux
from calculations.interface_solver import calculate_oxide_metal_system
from calculations.parallel_oxide_defect_paths import (calculate_parallel_path_flux,
                                                      calculate_parallel_path_flux_defective_metal)
from calculations.wall_labels import label_model1, label_model2
from calculations.permeability import UNITS_SIEVERTS, UNITS_HENRY
import pandas as pd
from IPython.display import display
from calculations.figures import results_dir, save_panels
from calculations import sweep_figures as sf
FIGS = results_dir("effect-of-pressure")                             # figures and fit tables go in this folder

SIM          = build_simulation_config()
T_op, P_down = CONDITIONS['T_operating'], SIM['P_downstream']
L_ox, L_m    = SIM['L_oxide'], SIM['L_metal']
P = np.logspace(*np.log10(CONDITIONS['P_range']), CONDITIONS['n_points'])   # upstream pressures (Pa)
P = P[P > P_down]                                                            # driving force must be positive

# Temperature is fixed, so the layer properties are computed once
OX = get_oxide_properties_at_T(SIM['oxide_name'], T_op); OX['thickness'] = L_ox
ME = get_metal_properties_at_T(SIM['metal_name'], T_op); ME['thickness'] = L_m

LEVELS = {   # level: (title, wall thickness for Φ_app)
    'L1':  ('Level 1: Perfect metal (lattice diffusion)',          L_m),
    'L2a': ('Level 2a: Perfect oxide alone (molecular H$_2$)',     L_ox),
    'L2b': ('Level 2b: Perfect oxide + perfect metal',             L_ox + L_m),
    'L3':  ('Level 3: Defective oxide + perfect metal',            L_ox + L_m),
    'L4':  ('Level 4: Defective metal (traps + grain boundaries)', L_m),
    'L5':  ('Level 5: Defective oxide + defective metal',          L_ox + L_m),
}

## 2. Physics (one solver call per level, then the sweep and the fit)

In [ ]:
def solve(level, Pk):
    """Flux and regime label for one level at one upstream pressure (T fixed at T_op)."""
    if level.endswith('L6'):
        return solve_m2(level, Pk)                       # Model 2 levels, defined further down
    ox, me = dict(OX), dict(ME)
    D, K  = me['D_metal'], me['K_s_metal']
    micro = dict(microstructure_params=MICROSTRUCTURE, lattice_density=MICROSTRUCTURE['N_L'], mode='both')
    if level == 'L1':  return calculate_simple_metal_flux(D, K, L_m, Pk, P_down)['flux'], label_model1('L1')
    if level == 'L2a': return molecular_diffusion_flux(ox['D_ox'], ox['K_ox'], L_ox, Pk, P_down), label_model1('L2a')
    if level == 'L2b': r = calculate_oxide_metal_system(Pk, P_down, ox, me)
    if level == 'L3':  r = calculate_parallel_path_flux(Pk, P_down, ox, me, OXIDE_DEFECTS)
    if level == 'L4':  r = calculate_defective_metal_flux(D_lattice=D, K_s=K, metal_thickness=L_m,
                                                          P_up=Pk, P_down=P_down, temperature=T_op, **micro)
    if level == 'L5':  r = calculate_parallel_path_flux_defective_metal(Pk, P_down, ox, me,
                                                                        OXIDE_DEFECTS, T_op, **micro)
    return r.get('flux_total', r.get('flux')), label_model1(level, r, Pk, P_down, ox, me, OXIDE_DEFECTS, T_op, micro)

def sweep(level):
    # Φ_app written out: apparent_permeability() takes one pressure, not an array of them
    J, regime = map(np.array, zip(*[solve(level, Pk) for Pk in P]))
    if level == 'L2a':                                   # Henry: J·L/ΔP
        return J, J * L_ox / (P - P_down), UNITS_HENRY, regime
    return J, J * LEVELS[level][1] / (np.sqrt(P) - np.sqrt(P_down)), UNITS_SIEVERTS, regime

def power_law(P, y):
    """y = A·P^n, fitted as ln y = ln A + n·ln P."""
    fit = stats.linregress(np.log(P), np.log(y))
    return fit.slope, np.exp(fit.intercept)                  # n, A

## 3. Figures

In [ ]:
# Figures: calculations/sweep_figures.py draws every figure of this notebook in the house style, one figure
# per panel (J and Φ_app), each written by calculations.figures.save_panels as <name>-<panel>.pdf, a 600 dpi
# .png and a _caption.tex whose numbers come from the results. Colour and marker mean the regime: colour = the
# limiting step, an open mark = a defect path carries most of the flux, a diamond = trap control
# (calculations/wall_labels.py). The fitted exponent of each regime is in the key below the plot.
from calculations.wall_labels import describe as look, fitted
ALLOY = sf.alloy_name(SIM['metal_name'])                # the alloy's name in the figure captions

## 4. Fits and figures

In [ ]:
def p_tag(Pc):
    """Short pressure label: 1e-07 -> '1e-7 Pa', 1.0 -> '1e0 Pa', 1e+07 -> '1e7 Pa'."""
    return f'{Pc:.0e}'.replace('e+0', 'e').replace('e-0', 'e-') + ' Pa'

def fit_table(rows):
    """Every fitted line as one row: shown under the figure, and returned for the CSV."""
    table = pd.DataFrame(rows).replace({r'\$_2\$': '₂'}, regex=True)
    with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 220,
                           'display.max_columns', 20):
        display(table)
    return table

EXPONENT = 'n (J) / slope (Φ_app)'                  # Φ_app ∝ P^(n − 0.5), or P^(n − 1) for L2a

def net_row(y, rows, quantity, tag):
    """The net exponent of the whole trend, Δln y / Δln P between the two ends of the range: the average
    of the local exponent d ln y / d ln P over the range, not a fitted line. A row of the table, never drawn."""
    ok = np.flatnonzero(np.isfinite(y) & (y > 0))
    a, b = ok[0], ok[-1]
    rows.append({'quantity': quantity, 'condition': tag, 'fit': 'net (whole range)',
                 'P window (Pa)': f'{P[a]:.1e}–{P[b]:.1e}', 'points': int(ok.size),
                 EXPONENT: np.log(y[b] / y[a]) / np.log(P[b] / P[a]), 'A (SI)': np.nan})

def regime_rows(y, regime, rows, quantity, tag):
    """One power-law fit per regime, each a row of the table; the figures draw the same fits."""
    for reg in dict.fromkeys(regime):
        m = regime == reg
        if not fitted(reg) or m.sum() < 3:
            continue                                     # drawn, not fitted
        n, A = power_law(P[m], y[m])
        rows.append({'quantity': quantity, 'condition': tag, 'fit': look(reg)[0],
                     'P window (Pa)': f'{P[m].min():.1e}–{P[m].max():.1e}', 'points': int(m.sum()),
                     EXPONENT: n, 'A (SI)': A})

def pressure_figure(level, curves=None, temperatures=None):
    """The log-log figures of one level, J and Φ_app (one figure each, regime-resolved fits, the reference
    slopes n = 1 and 0.5 on J), and the fit table, which also holds the net n of the whole range.
    curves = [(tag, sweep output), ...], one per condition, at `temperatures` (default: T_op).
    Returns (panels, fit table)."""
    curves = curves or [(f'{T_op:.0f} K', sweep(level))]
    temperatures = temperatures or [T_op] * len(curves)
    units = curves[0][1][2]
    rows = []
    for r, qty in enumerate(('J', 'Phi_app')):
        for tag, out in curves:                         # out = (J, Φ_app, units, regime)
            net_row(out[r], rows, qty, tag)
        for tag, out in curves:
            regime_rows(out[r], out[3], rows, qty, tag)
    three = len(curves) > 1
    panels = sf.sweep_panels('pressure', level,
                             [sf.curve(tag, P, out[0], out[1], out[3], Tc, P, L_ox, L_m)
                              for (tag, out), Tc in zip(curves, temperatures)],
                             units, P_down, variant='3T' if three else 'op', alloy=ALLOY)
    return sf.show(panels), fit_table(rows)

## 5. Model 1 (L1–L5)

In [ ]:
for level in LEVELS:
    panels, fits = pressure_figure(level)
    save_panels(panels, f"{level}_pressure", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_pressure_fits.csv", index=False)

## 6. Model 2 (L6 family)

In [ ]:
# =============================================================================
# MODEL 2 (L6 family): dissociation on the gas-facing surface, atomic H in oxide and metal
# Reuses solve() -> sweep() -> pressure_figure() above; only the solver is new.
# =============================================================================
from calculations.surface_kinetics import (get_all_properties, solve_steady_state_flux_L1L6,
                                           solve_steady_state_flux_L2aL6, solve_steady_state_flux_direct,
                                           calculate_mixed_defect_flux_L6, calculate_defective_metal_flux_L6,
                                           calculate_full_model_flux_L346_v2)
from calculations.config.model_config import METALS, get_surface_kinetics_from_config

MICRO   = SIM['microstructure']
DEFECTS = SIM['defect_config']
SIEVERTS_PINHOLE = SIM['oxide_defects'].get('use_sieverts_pinhole', False)

# Temperature is fixed, so the Model 2 properties are computed once
PROPS = get_all_properties(SIM['oxide_name'], SIM['metal_name'], T_op)       # oxide surface + transport
SM    = get_surface_kinetics_from_config(SIM['metal_name'], T_op, METALS)    # bare-metal surface

LEVELS_M2 = {   # level: (title, wall thickness for Φ_app) -- every Model 2 level is Sieverts
    'L1L6':  ('Level 1+6: Perfect metal + surface kinetics',                     L_m),
    'L2aL6': ('Level 2a+6: Perfect oxide alone + surface kinetics',              L_ox),
    'L2bL6': ('Level 2b+6: Perfect oxide + perfect metal + surface kinetics',    L_ox + L_m),
    'L3L6':  ('Level 3+6: Defective oxide + perfect metal + surface kinetics',   L_ox + L_m),
    'L4L6':  ('Level 4+6: Perfect oxide + defective metal + surface kinetics',   L_ox + L_m),
    'L5L6':  ('Level 5+6: Defective oxide + defective metal + surface kinetics', L_ox + L_m),
}
LEVELS.update(LEVELS_M2)

def solve_m2(level, Pk):
    """Flux and regime label for one Model 2 level at one pressure, e.g. 'oxide surface' or 'pinholes/metal diffusion'."""
    k, Keq, Dox, Kox, Dm, Ksm = (PROPS[key] for key in ('k_diss', 'K_eq', 'D_ox', 'K_ox', 'D_m', 'K_s_m'))
    pin  = {} if SIEVERTS_PINHOLE else dict(k_diss_metal=SM['k_diss'], K_eq_metal=SM['K_eq'])
    mode = MICRO.get('mode', 'both')
    if level == 'L1L6':  r = solve_steady_state_flux_L1L6(Pk, P_down, L_m, SM['k_diss'], SM['K_eq'], Dm, Ksm)
    if level == 'L2aL6': r = solve_steady_state_flux_L2aL6(Pk, P_down, k, Keq, Dox, Kox, L_ox)
    if level == 'L2bL6': r = solve_steady_state_flux_direct(Pk, P_down, L_m, k, Keq, Dox, Kox, L_ox, Dm, Ksm)
    if level == 'L3L6':  r = calculate_mixed_defect_flux_L6(Pk, P_down, L_m, k, Keq, Dox, Kox, L_ox,
                                                            Dm, Ksm, DEFECTS, **pin)
    if level == 'L4L6':  r = calculate_defective_metal_flux_L6(Pk, P_down, L_m, T_op, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, microstructure_params=MICRO, mode=mode)
    if level == 'L5L6':  r = calculate_full_model_flux_L346_v2(Pk, P_down, L_m, T_op, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, MICRO, DEFECTS, mode=mode, **pin)
    return r.get('J_total', r.get('J_ss', r.get('flux'))), label_model2(level, r, DEFECTS)

for level in LEVELS_M2:
    panels, fits = pressure_figure(level)
    save_panels(panels, f"{level}_pressure", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_pressure_fits.csv", index=False)

## 7. Three temperatures: the pressure sweep at the low / mid / high of `T_range`

In [ ]:
# =============================================================================
# THREE TEMPERATURES: the pressure sweep at the low / mid / high of T_range
# Same figures as above, with one curve per temperature. Colour and marker still mean the regime;
# the name at the end of each curve and the line style of its fits mean the temperature.
# =============================================================================
from contextlib import contextmanager

T_lo, T_hi = CONDITIONS['T_range']
T_SET = [T_lo, (T_lo + T_hi) / 2, T_hi]             # low / mid / high: 573, 923, 1273 K

@contextmanager
def at_temperature(Tc):
    """Run solve() and sweep() above at another temperature; T_op and its properties are restored afterwards."""
    global T_op, OX, ME, PROPS, SM
    saved = T_op, OX, ME, PROPS, SM
    T_op  = Tc
    OX    = dict(get_oxide_properties_at_T(SIM['oxide_name'], Tc), thickness=L_ox)
    ME    = dict(get_metal_properties_at_T(SIM['metal_name'], Tc), thickness=L_m)
    PROPS = get_all_properties(SIM['oxide_name'], SIM['metal_name'], Tc)
    SM    = get_surface_kinetics_from_config(SIM['metal_name'], Tc, METALS)
    try:
        yield
    finally:
        T_op, OX, ME, PROPS, SM = saved

for level in LEVELS:
    curves = []
    for Tc in T_SET:
        with at_temperature(Tc):
            curves.append((f'{Tc:.0f} K', sweep(level)))
    panels, fits = pressure_figure(level, curves, temperatures=T_SET)
    save_panels(panels, f"{level}_pressure_3T", FIGS, module=sf.MODULE)
    fits.to_csv(FIGS / f"{level}_pressure_3T_fits.csv", index=False)